In [ ]:
import pandas as pd
import numpy as np
import gdown
from scipy.sparse import csr_matrix
import sys
import os
import platform
import json
import time
from pathlib import Path
from datetime import datetime, timezone
import torch

project_root = Path.cwd()
if not (project_root / "src" / "recommendationsystems").exists():
    project_root = Path("/home/sagemaker-user/Recommendation_systems")
sys.path.insert(0, str(project_root / "src"))

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
results_dir = project_root / "results" / run_id
results_dir.mkdir(parents=True, exist_ok=True)
prep_times = {}
comparison_results = []

def synchronize_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def timed_call(func, *args, **kwargs):
    synchronize_device()
    started = time.perf_counter()
    value = func(*args, **kwargs)
    synchronize_device()
    return value, time.perf_counter() - started

def prediction_latency(func, filter_seen, repeats=100):
    # Warm up first; time individual requests, not batched inference.
    users = eval_users[:min(repeats, len(eval_users))]
    if not users:
        raise ValueError("No evaluation users")
    func(users[0], k=10, filter_seen=filter_seen)
    timings = [timed_call(func, user, k=10, filter_seen=filter_seen)[1] for user in users]
    return float(np.mean(timings)), float(np.median(timings)), len(timings)

def save_results():
    frame = pd.DataFrame(comparison_results)
    frame.to_csv(results_dir / "metrics.csv", index=False)
    return frame

try:
    import psutil
    ram_bytes = psutil.virtual_memory().total
except ImportError:
    try:
        ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (ValueError, OSError, AttributeError):
        ram_bytes = None
hardware = {
    "platform": platform.platform(), "cpu": platform.processor(),
    "logical_cpus": os.cpu_count(), "ram_bytes": ram_bytes,
    "python": platform.python_version(), "torch": torch.__version__,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(results_dir / "hardware.json").write_text(json.dumps(hardware, indent=2))
print("Results directory:", results_dir)


**1. Pobieranie danych**

In [3]:
ID_TRAIN = "1zjZe-u52WdE4obfOj8c1foCysYuJvfjB"
ID_TEST = "1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK"

print("Pobieranie zbioru treningowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TRAIN}", "df_train.csv", quiet=False)

print("Pobieranie zbioru testowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TEST}", "df_test.csv", quiet=False)

# Wczytanie zbiorów
df_train = pd.read_csv("df_train.csv")
df_test = pd.read_csv("df_test.csv")

print(f"Dane treningowe: {len(df_train)} wierszy, dane testowe: {len(df_test)} wierszy")

Pobieranie zbioru treningowego:


Downloading...
From: https://drive.google.com/uc?id=1zjZe-u52WdE4obfOj8c1foCysYuJvfjB
To: /content/df_train.csv
100%|██████████| 64.5M/64.5M [00:00<00:00, 91.3MB/s]


Pobieranie zbioru testowego:


Downloading...
From: https://drive.google.com/uc?id=1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK
To: /content/df_test.csv
100%|██████████| 11.7M/11.7M [00:00<00:00, 62.3MB/s]


Dane treningowe: 2076316 wierszy, dane testowe: 377262 wierszy


Jak wyglądają dane?

In [4]:
df_train.head()

,user,item,event,timestamp
0,1001518,94832,click,1581465600
1,2062683,52240,click,1581465600
2,2726272,135586,click,1581465600
3,1666865,4593,click,1581465601
4,129722,69676,click,1581465603


**2. Tworzenie słowników i par bezpośrednich przejść (item_t -> item_{t+1}) dla każdego użytkownika**

In [ ]:
_prep_started = time.perf_counter()
unique_items = df_train['item'].unique()
item2idx = {item: idx for idx, item in enumerate(unique_items)}
idx2item = {idx: item for item, idx in item2idx.items()}
n_items = len(unique_items)

train_sorted = df_train.sort_values(by=['user', 'timestamp'])
train_sorted['next_item'] = train_sorted.groupby('user')['item'].shift(-1)
transitions = train_sorted.dropna(subset=['next_item'])
prep_times[6] = time.perf_counter() - _prep_started


In [6]:
print(f"Liczba produktów: {n_items}")
print(transitions.head())

Liczba produktów: 13317
        user    item  event   timestamp  next_item
839633     4    7525  click  1581823094    65386.0
840230     4   65386  click  1581823257   169493.0
840770     4  169493  click  1581823401   115778.0
840837     4  115778  click  1581823419    23751.0
841195     4   23751  click  1581823517   181168.0


In [ ]:
_prep_started = time.perf_counter()
row_indices = transitions['item'].map(item2idx).values
col_indices = transitions['next_item'].map(item2idx).values
prep_times[8] = time.perf_counter() - _prep_started


Znalezienie par bezpośrednich przejść (item_t -> item_{t+1}) dla każdego użytkownika i indeksy macierzy przejścia.

In [ ]:
_prep_started = time.perf_counter()
train_sorted['next_item'] = train_sorted.groupby('user')['item'].shift(-1)
transitions = train_sorted.dropna(subset=['next_item'])

row_indices = transitions['item'].map(item2idx).values
col_indices = transitions['next_item'].map(item2idx).values
prep_times[10] = time.perf_counter() - _prep_started


**3. Tworzenie macierzy zliczeń przejść i normalizacja wierszy do prawdopodobieństw P(B|A)**

In [ ]:
_prep_started = time.perf_counter()
counts_matrix = csr_matrix(
    (np.ones(len(row_indices)), (row_indices, col_indices)),
    shape=(n_items, n_items),
    dtype=np.float32
)

row_sums = np.array(counts_matrix.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
transition_matrix = counts_matrix.multiply(1.0 / row_sums[:, np.newaxis]).tocsr()
prep_times[12] = time.perf_counter() - _prep_started


**3. Ostatnia kliknięta oferta z train dla danego użytkownika i historia**

In [ ]:
_prep_started = time.perf_counter()
last_train_items = train_sorted.groupby('user')['item'].last().to_dict()

user_train_history = df_train.groupby('user')['item'].apply(set).to_dict()
prep_times[14] = time.perf_counter() - _prep_started


Globalna lista najpopularniejszych ofert

In [ ]:
_prep_started = time.perf_counter()
popular_items_train = df_train['item'].value_counts().index.tolist()
prep_times[16] = time.perf_counter() - _prep_started


**4. Generowanie rekomendacji**

In [12]:
def recommend_mcrs(user, k=10, filter_seen=True):
    last_item = last_train_items.get(user)
    seen_items = user_train_history.get(user, set()) if filter_seen else set()

    recs = []
    if last_item in item2idx:
        item_idx = item2idx[last_item]
        probs = transition_matrix[item_idx].toarray().flatten()

        if probs.sum() > 0:
            top_indices = np.argsort(probs)[::-1]
            for idx in top_indices:
                candidate = idx2item[idx]
                if candidate not in seen_items:
                    recs.append(candidate)
                if len(recs) == k:
                    return recs

    #dopełnienie do k najpopularniejszymi ofertami z treningowych
    for item in popular_items_train:
        if item not in seen_items and item not in recs:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

**5. Ewaluacja (precision + recall)**

BASELINE:

In [13]:
def recommend_baseline(user, k=10, filter_seen=True):
    seen_items = user_train_history.get(user, set()) if filter_seen else set()
    recs = []
    for item in popular_items_train:
        if item not in seen_items:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

Użytkownicy w zbiorach:

In [ ]:
train_users = set(df_train['user'].unique())
test_users = set(df_test['user'].unique())
eval_users = sorted(train_users.intersection(test_users))
data_info = {
    "train_interactions": len(df_train), "test_interactions": len(df_test),
    "users": len(train_users | test_users),
    "items": len(set(df_train["item"]) | set(df_test["item"])),
    "evaluation_users": len(eval_users),
}
(results_dir / "dataset.json").write_text(json.dumps(data_info, indent=2))


In [ ]:
test_true_inter = df_test.groupby('user')['item'].apply(set).to_dict()

def evaluate(model_func, users, k=10, filter_seen=True):
    if k <= 0:
        raise ValueError("k must be positive")
    precisions = []
    recalls = []

    for user in users:
        actual = test_true_inter.get(user, set())
        if not actual:
            continue

        predicted = model_func(user, k=k, filter_seen=filter_seen)

        hits = len(set(predicted).intersection(actual))
        precisions.append(hits / k)
        recalls.append(hits / len(actual))

    return np.mean(precisions), np.mean(recalls)

Ewaluacja dla Łańcuchów Markova i Baseline

In [ ]:
comparison_results = []
# MCRS includes sorting, transition matrix, histories and popularity fallback.
# Popularity includes its own history preparation measured separately below.
(_, baseline_history_s) = timed_call(lambda: df_train.groupby("user")["item"].apply(set).to_dict())
train_times = {
    "MCRS": sum(prep_times.values()),
    "Popularity": prep_times[16] + baseline_history_s,
}
for filter_seen in (True, False):
    for name, model_func in (("Popularity", recommend_baseline), ("MCRS", recommend_mcrs)):
        (precision, recall), evaluation_s = timed_call(
            evaluate, model_func, eval_users, k=10, filter_seen=filter_seen
        )
        mean_s, median_s, prediction_samples = prediction_latency(model_func, filter_seen)
        comparison_results.append({
            "method": name, "filter_seen": filter_seen,
            "Precision@10": precision, "Recall@10": recall,
            "train_s": train_times[name], "evaluation_s": evaluation_s,
            "prediction_mean_s": mean_s, "prediction_median_s": median_s,
            "prediction_samples": prediction_samples,
            "evaluation_mode": "individual", **data_info,
        })
        display(save_results())


## 6. SASRec

Implementacja architektury Self-Attentive Sequential Recommendation. Model jest uczony wyłącznie na chronologicznych sekwencjach treningowych, a ewaluacja używa tego samego zbioru użytkowników oraz definicji Precision@10 i Recall@10 co MCRS.

In [ ]:
from recommendationsystems.sasrec import fit_sasrec, precision_recall_at_k

(sasrec, sasrec_losses), sasrec_train_s = timed_call(
    fit_sasrec,
    df_train,
    epochs=10,
    batch_size=512,
    max_len=50,
    hidden_dim=64,
    num_blocks=2,
    num_heads=1,
    dropout=0.2,
    seed=42,
)
pd.DataFrame({"epoch": range(1, len(sasrec_losses) + 1), "loss": sasrec_losses}).to_csv(results_dir / "sasrec_losses.csv", index=False)
print(f"SASRec training: {sasrec_train_s:.3f} s")


In [ ]:
# Train once, then evaluate the same model in both modes.
# Remove earlier SASRec rows if this evaluation cell is run again.
comparison_results = [row for row in comparison_results if row["method"] != "SASRec"]
for filter_seen in (True, False):
    (precision, recall, count), evaluation_s = timed_call(
        precision_recall_at_k, sasrec, df_test,
        users=eval_users, k=10, filter_seen=filter_seen
    )
    mean_s, median_s, prediction_samples = prediction_latency(sasrec.recommend, filter_seen)
    comparison_results.append({
        "method": "SASRec", "filter_seen": filter_seen,
        "Precision@10": precision, "Recall@10": recall,
        "train_s": sasrec_train_s, "evaluation_s": evaluation_s,
        "prediction_mean_s": mean_s, "prediction_median_s": median_s,
        "prediction_samples": prediction_samples,
        "evaluation_mode": "batched_512", **data_info,
    })
    display(save_results())
print("Saved:", results_dir / "metrics.csv")
